# ???????????? ???????

## ??????????

1. ?????????? ??????
2. ???????? ? ???????? ????? ??????????
3. ????????? ?????? ?? ???????
4. ???????????? ??????? ????????? ????????
* 4.1  ???????? ??????
* 4.2  ????????? ???
* 4.3  ????????????? ?????????
5. ???????? ?????? ?? ???????? ???????
6. ???????? ?????? ?? ????????????
7. ????? ????? ?? ???????

## 1 ?????????? ??????

? ????? ???????????? ?????? ? ????????? ????????, ??????? ??? ??????? ?? ??? ?????? (?? ??????? ????? ??????????????? ?????? ???????). ????? ????????? ?????? ??? ?????? ?????????????, ??????? ??????? ?????????? ?????. ????????????? ?????? ?? ??????????? ? ?? ?? ??? ???????.

????????? ?????? ? ??????????? ??????? ????????? *accuracy*. ????? ????? ?????? ???????, ????? ??????? ???? ?????????? ??????? ?? ??????? ???? ?? 0.75. ????????? *accuracy* ?? ???????? ??????? ??????????????.

**???????? ??????**

?????? ?????? ? ?????? ?????? ? ??? ?????????? ? ????????? ?????? ???????????? ?? ?????. ????????:
* *?alls* ? ?????????? ???????,
* *minutes* ? ????????? ???????????? ??????? ? ???????,
* *messages* ? ?????????? sms-?????????,
* *mb_used* ? ??????????????? ????????-?????? ? ??,
* *is_ultra* ? ????? ??????? ??????????? ? ??????? ?????? (???????? ? 1, ??????? ? 0).

## 2 ???????? ? ???????? ????? ??????????

In [ ]:
import pandas as pd
from IPython.display import display #??????????? ????? ??????? ??? ????????? ?????? ?????? (?? ???????? print)
pd.set_option('display.max_columns', None) #????????? ??????????? ?????? ???? ???????? ?? ?????
from sklearn.tree import DecisionTreeClassifier #???????? ??????
from sklearn.model_selection import train_test_split # ??????? ?? 2 ???????
from sklearn.ensemble import RandomForestClassifier # ????????? ???
from sklearn.linear_model import LogisticRegression # ????????????? ?????????
from sklearn.metrics import accuracy_score # ???????? ????????

In [ ]:
df = pd.read_csv('users_behavior.csv', sep=',')

print(df.info())

display(df.head())

df.describe()

In [ ]:
df['is_ultra'].value_counts()

*?? ???????? ???????? ?????, ??? ????? ? ??? 3214 ????? ? 5 ????????. ??????????? ?????? ???????????. ????? describe ???????, ??? ?????? ??? ??????????, ???????? ??????????? ???, ?.?. ??? ???? 4-? ?????????? ??????? ???????? ?????? ? ?????????. ?? ?????? "??????" ? ??? 985 ?????????????, ????????? ?? "??????". ?? ???????? ?????? ????????????? ?????? ?? ?????????, ????? ?????????? ? ????????? ?????? ?? ???????.*

## 3 ????????? ?????? ?? ???????

??? ???? ??????? ???? ??????? ?? ?????????, ????????????? ? ???????? ???????. ???????? ?? ? ?????????? ??????????? 60/20/20. ??? ????? ???????? 2 ???? ????? train_test_split. ? ?????? ???????? ??????? ????????? ??????? ?? ????????????? ? ????????, ?? ?????? ???????? ????????????? ? ????????.

In [ ]:
df_train, df_valid_test = train_test_split(df, test_size=0.4, random_state=12345) # ????????? ??????? ? ????????? 4 ? 10
df_valid, df_test = train_test_split(df_valid_test, test_size=0.5, random_state=12345) # ????????? ???-?? ? ???????? ???????

# ???????? ???????????? ??????? ???????
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

1928 / 3214 ? 60 %, 643 / 3214 ? 20 %. ????????? ????????? ?????????. 

## 4 ???????????? ??????? ????????? ????????

????????? 3 ?????? ????????? ???????? ??? ?????? ????????????? (???????? ??????, ????????? ??? ? ????????????? ?????????) ? ??????? ?????? ??? ?????????? ???????? ?? ???????? ???????. ????? ????????? ? ????????? ?????? ??????? ??????? ???????? ? ??????? ??????? ????????? ? ????????????? ???????.

In [ ]:
train_features = df_train.drop(['is_ultra'], axis=1) # ??????? ???????? ??? ????????? ???????
train_target = df_train['is_ultra'] # ??????? ??????? ??????? ??? ????????? ???????
valid_features = df_valid.drop(['is_ultra'], axis=1) # ??????? ???????? ??? ????????????? ???????
valid_target = df_valid['is_ultra'] # ??????? ??????? ??????? ??? ????????????? ???????

### 4.1 ???????? ??????

In [ ]:
%%time
best_model = None
best_result = 0
best_depth = 0
for depth in range(1, 10):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth) # ??????? ?????? ? ???????? ???????? ??????
    model.fit(train_features, train_target) # ??????? ?????? ?? ????????? ???????
    predictions = model.predict(valid_features) # ??????? ???????????? ?????? ?? ????????????? ???????
    result = accuracy_score(valid_target, predictions) # ????????? ???????? ??????
    if result > best_result:
        best_model = model
        best_result = result
        best_depth = depth
print('????????? ???????? ?????? {:.3f} ??? ??????? ?????? {}'.format(best_result, best_depth))

### 4.2 ????????? ???

In [ ]:
%%time
best_model = None
best_result = 0
best_est = 0
best_depth = 0
best_split = 0
for est in range(10, 51, 10):
    for depth in range (1, 9):
        for spl in (2, 5):
            model = RandomForestClassifier(random_state=12345, 
                                     n_estimators=est, 
                                     max_depth=depth, min_samples_split = spl) # ?????????????? ??????
            model.fit(train_features, train_target) # ?????? ?????? ?? ????????????? ???????
            result = model.score(valid_features, valid_target) # ????????? ???????? ?????? ?? ????????????? ???????
            if result > best_result:
                best_model = model
                best_result = result
                best_est = est
                best_depth = depth
                best_split = spl

print("Accuracy ????????? ?????? ?? ????????????? ???????:", best_result, 
      "?????????? ????????:", best_est, "???????????? ???????:", best_depth,
     "??????????? ?????????? ???????? ??? ??????????", best_split)

### 4.3 ????????????? ?????????

In [ ]:
%%time
model = LogisticRegression(random_state=12345) # ?????????????? ?????? ????????????? ????????? ? ?????????? random_state=12345
model.fit(train_features, train_target) # ?????? ?????? ?? ????????????? ???????
result = model.score(valid_features, valid_target) # ??????? ??????? ???????? ?????? ?? ????????????? ???????

print("Accuracy ?????? ????????????? ????????? ?? ????????????? ???????:", result)

????, ?? ?????, ??? ????????? ???????? ?????? (0,809) ?? ????????????? ??????? ???????? ??? ?????????? ????????? "????????? ???", ???? ???? ? "????????? ??????" - 0.785; ? ? ????? ?????? "????????????? ?????????" - 0.758. ???????? ??????????? ??????? ???????? ?????? 0.75., ?????? ????? ?????? ????? ???? ??????? ??? ???????? ?? ???????? ???????. ??????? ?????? ? ????????? ?????????, ?.?. "????????? ???" ? ????????????? ???????????? ???????????: ?????????? ???????? 40, ??????? 7, ??????????? ?????????? ???????? ??? ?????????? 2.

## 5 ???????? ?????? ?? ???????? ???????

In [ ]:
test_features = df_test.drop(['is_ultra'], axis=1) # ??????? ???????? ??? ???????? ???????
test_target = df_test['is_ultra'] # ??????? ??????? ??????? ??? ???????? ???????

model_forest = RandomForestClassifier(random_state=12345, n_estimators=40, max_depth=7, min_samples_split = 2) # ?????????????? ??????
model_forest.fit(train_features, train_target) # ?????? ?????? ?? ????????????? ???????
forest_test_result = model_forest.score(test_features, test_target) # ????????? ???????? ?????? ?? ???????? ???????
print("Accuracy ?????? ?????????? ???? ?? ???????? ???????:", forest_test_result)

???????? ?????????? ??????? ???????? (0,803) ?? ???????? ???????, ??? ???????????? ????????????????? ?????????? ????????? ? ??? ?????????.

## 6 ???????? ?????? ?? ????????????

???????? *Accuracy* ???????? ????? ???????????? ???????????, - ??? ????? ???????? ????????. ? ???????, ???? ? ??????? 950 ????????????? ? 50 ????????????? ????????, ?? ??? ????????? ????????? ????????????? ?? ??????? ???? ?????????? ??????? 0.95. ??? ????????, ??? ???? ????????????? ??????? ???? ?? ???? ?? ????? ???????? ?????????? ? ???????? ?????? ?????????, ? ?????? ? ???? ??????? ????????????? ? ?????? ???????. 

????? ????????? ? ????? ???????? ?????????? ?????? ?????? ????????? ??? ???????? ???? ?????? ? ???????? ?????? ????????????? ? *confusion matrix* (??????? ??????). 
- ???? ???????? ?????????? "1" ? ??? ????????? ?????, ?? ????? ??????? ???????? True Positive (?????? ?????????????) ??? TP. 
- ???? ???????? ?????????? "1" ? ??? ????????? ???????, ?? ????? ??????? ???????? False Positive (?????? ?????????????) ??? FP.
- ???? ???????? ?????????? "0" ? ??? ????????? ?????, ?? ????? ??????? ???????? True Negative (?????? ?????????????) ??? TN. 
- ???? ???????? ?????????? "0" ? ??? ????????? ???????, ?? ????? ??????? ???????? False Negative (?????? ?????????????) ??? FN.

???????? Accuracy ? ??????? ??????? ?????? ????????? ?? ????????? ???????:

$$Accuracy = \frac{TP+TN}{TP+TN+FP+FN}$$

??? ???? ??????? ????, ??????? accuracy ????? ???? ??????????? ? ??????? ? ????????? ????????. ??? ?????? ???????? ?????? ????????? ?? ?????? ?? ??????? ?? ??????????? ?????? ??????? *precision* (????????) ? *recall* (???????).

$$Presision = \frac{TP}{TP+FP}$$

$$Recall = \frac{TP}{TP+FN}$$

???????? ??????????, ????? ???? ????????, ?????????? ??????????????? ??? ?????????????, ?????????????? ???????? ??????????????.

??????? ??????????, ????? ????? ????????????? ???????? ???? ???????? ???????????????.

??? ?????? ????????? ??????? ???????? ?????? ???????????? ???????????? ?? ??????? - F-????, ??????? ???????? ??????? ????????????? precision ? recall. F-???? ??????????? ?? ???????:

$$F = (1+\beta^2)*\frac{presision*recall}{(\beta^2*presision)+recall}$$

$\beta$ ? ?????? ?????? ?????????? ??? ???????? ? ???????.

F-???? ????????? ????????? ??? ??????? ? ????????, ??????? ???????, ? ?????? ? ????, ???? ???? ?? ?????????? ?????? ? ????.

? sklearn ???? ??????? ??????? **_metrics.classificationreport**, ???????????? *recall*, *precision* ? *F-????* ??? ??????? ?? ???????, ? ????? ?????????? ??????????? ??????? ??????. ??????? ??? ??????? ??? ?????????? ??????.

In [ ]:
from sklearn.metrics import classification_report # ??????????? ??????? ?????? ?? ?????????????
report = classification_report(test_target, model_forest.predict(test_features), target_names=['Smart', 'Ultra'])
print(report)

????, ?? ?????, ??? ????????? ?????? ????? ????????? ?????????? ???????????? ?????? "0", ??????? ????????????? ?????? "?????". ??? 3 ??????? ??? ??????? ?????? ?? ?????????? ??????? ??????. ????? ????????? ?????? ?????????? ???????????? ??? ?????? "1", ??????? ????????????? ?????? "??????". ?? ???????? ???????????? ???????? ??????? recall, ??????? ??? ??????? ?????? ??????? ???????? 0.51.

## 7 ????? ????? ?? ???????

? ???? ???????? ???? ????????? ?????????? ????????? ???????? ??? ???????????? ?????? ????????????? ????????? ???????? ???????? "????????? ???" ?? ?????????? ???????????: ?????????? ???????? 40, ??????? 7, ??????????? ?????????? ???????? ??? ?????????? 2. 
    
?????? ???????? ???????? ???????? ???????????? ???? ?????? ??????? ??? ?? ????????????? (0,805), ??? ? ?? ???????? (0,803) ????????. ?????? ???????????? ??????????? ?????? ????????, ??? ???????? ?????? ??????? ?????? ??????????? ? ????????????? ?????? ?????? "0" (*precision = 0.81, recall = 0.94, f1 = 0.87*) ? ???? ? ????????????? ?????? ?????? "1" (*precision = 0.79, recall = 0.51, f1 = 0.62*).
    